In [ ]:
import numpy as np

In [ ]:
rT = (np.array([1, 0, 1]))/(np.sqrt(2))

def measure(r_true, axis, n, rng):   # axis is 0 as X, 1 as Y, and 2 as Z
    P1 = (1 - r_true[axis]) / 2      # probability of measuring 1
    P0 = (1 + r_true[axis]) / 2      # probability of measuring 0 (unused, kept for reference)
    u = rng.random(n)
    ones = 0
    zeros = 0
    for a in u:
        if (a < P1):
            ones = ones + 1
        if (a >= P1):
            zeros = zeros + 1
    return ones, zeros

def reconstruct_rho(x_1, x_0, y_1, y_0, z_1, z_0):
    rz = ((-1)*z_1 + z_0) / (z_0 + z_1)
    ry = ((-1)*y_1 + y_0) / (y_0 + y_1)
    rx = ((-1)*x_1 + x_0) / (x_0 + x_1)

    # only rescale if the estimate landed outside the Bloch sphere
    norm = np.sqrt(rz**2 + rx**2 + ry**2)
    if norm > 1:
        norm_factor = 1 / norm
    else:
        norm_factor = 1

    rz = rz * norm_factor
    ry = ry * norm_factor
    rx = rx * norm_factor

    p = rho(rx, ry, rz)
    return p
def rho (rx, ry, rz):
  p = 0.5 * (np.array([[1 + rz, rx - ry*(1j)], [rx + ry*(1j), 1 - rz]]))
  return p
states = {
    "ideal":        1.0 * rT,
    "depol 0.8":    0.8 * rT,
    "tilt in-plane": 0.9 * np.array([np.sin(np.pi/4+0.2), 0, np.cos(np.pi/4+0.2)]),
    "tilt out":     0.9 * np.array([np.cos(0.3)*rT[0], np.sin(0.3), np.cos(0.3)*rT[2]]),
    "mixed":        np.array([0.0, 0.0, 0.0]),
    "zero state":   np.array([0.0, 0.0, 1.0]),
}

# GRADER USE ONLY

def traceDistance (p_r, p_e):
  eigs = np.linalg.eigvalsh(p_e - p_r)
  return 0.5 * np.sum(np.abs(eigs))

rng = np.random.default_rng(seed=0)

for name, r_true in states.items():
    # GRADER ONLY: build the true matrix here, never inside reconstruct_rho
    p_true = rho(r_true[0], r_true[1], r_true[2])

    for n in (20, 200, 2000):
        dists, valid, rescaled = [], True, 0
        for _ in range(500):
            z_1, z_0 = measure(r_true, 2, n, rng)
            x_1, x_0 = measure(r_true, 0, n, rng)
            y_1, y_0 = measure(r_true, 1, n, rng)
            p_est = reconstruct_rho(x_1, x_0, y_1, y_0, z_1, z_0)
            ev = np.linalg.eigvalsh(p_est)
            valid = valid and abs(np.trace(p_est) - 1) < 1e-9 and ev.min() > -1e-9
            rescaled += ev.min() < 1e-9      # rescaled estimates land on the surface
            dists.append(traceDistance(p_true, p_est))
        print(name, n, round(np.mean(dists), 4), valid, rescaled/500)

ideal 20 0.1257 True 0.618
ideal 200 0.0421 True 0.52
ideal 2000 0.0133 True 0.564
depol 0.8 20 0.1513 True 0.204
depol 0.8 200 0.0499 True 0.0
depol 0.8 2000 0.0157 True 0.0
tilt in-plane 20 0.1346 True 0.374
tilt in-plane 200 0.0473 True 0.026
tilt in-plane 2000 0.0147 True 0.0
tilt out 20 0.1392 True 0.398
tilt out 200 0.0489 True 0.038
tilt out 2000 0.015 True 0.0
mixed 20 0.1803 True 0.0
mixed 200 0.0565 True 0.0
mixed 2000 0.0176 True 0.0
zero state 20 0.136 True 1.0
zero state 200 0.0444 True 1.0
zero state 2000 0.0137 True 1.0


In [ ]:
# ---------- SIMULATOR (the only part that knows r_true) ----------
def measure(r_true, axis, n, rng):
    """axis: 0 = X, 1 = Y, 2 = Z. Returns (number of 1s, number of 0s)."""
    P1 = (1 - r_true[axis]) / 2
    ones = int(np.sum(rng.random(n) < P1))
    return ones, n - ones


# ---------- PROTOCOL: Strategy A (Level 1 rho, then fidelity) ----------
def reconstruct_rho(x_1, x_0, y_1, y_0, z_1, z_0):
    rz = ((-1)*z_1 + z_0) / (z_0 + z_1)
    ry = ((-1)*y_1 + y_0) / (y_0 + y_1)
    rx = ((-1)*x_1 + x_0) / (x_0 + x_1)

    norm = np.sqrt(rz**2 + rx**2 + ry**2)
    norm_factor = 1 / norm if norm > 1 else 1

    rz, ry, rx = rz * norm_factor, ry * norm_factor, rx * norm_factor
    return 0.5 * np.array([[1 + rz, rx - ry*(1j)], [rx + ry*(1j), 1 - rz]])

c, s = np.cos(np.pi/8), np.sin(np.pi/8)
T = np.array([c, s])

def fidelity_A(x_1, x_0, y_1, y_0, z_1, z_0):
    p = reconstruct_rho(x_1, x_0, y_1, y_0, z_1, z_0)
    return np.real(T.conj() @ p @ T)


# ---------- EXPERIMENT (grader) ----------
rT = np.array([1, 0, 1]) / np.sqrt(2)
rng = np.random.default_rng(seed=0)

for k in (0.9, 1.0):
    r_true = k * rT
    F_true = 0.5 * (1 + r_true @ rT)
    for N in (60, 600, 6000):
        n = N // 3
        F_hats = []
        for _ in range(1000):
            x_1, x_0 = measure(r_true, 0, n, rng)
            y_1, y_0 = measure(r_true, 1, n, rng)
            z_1, z_0 = measure(r_true, 2, n, rng)
            F_hats.append(fidelity_A(x_1, x_0, y_1, y_0, z_1, z_0))
        print(f"k={k} N={N:5d} | bias {np.mean(F_hats)-F_true:+.4f}  SD {np.std(F_hats):.4f}")

k=0.9 N=   60 | bias -0.0204  SD 0.0637
k=0.9 N=  600 | bias +0.0002  SD 0.0264
k=0.9 N= 6000 | bias +0.0001  SD 0.0085
k=1.0 N=   60 | bias -0.0407  SD 0.0485
k=1.0 N=  600 | bias -0.0114  SD 0.0146
k=1.0 N= 6000 | bias -0.0033  SD 0.0047


In [ ]:
# ---------- SIMULATOR (the only part that knows r_true) ----------
def measure(r_true, axis, n, rng):
    """axis: 0 = X, 1 = Y, 2 = Z. Returns (number of 1s, number of 0s)."""
    P1 = (1 - r_true[axis]) / 2
    ones = int(np.sum(rng.random(n) < P1))
    return ones, n - ones


# ---------- PROTOCOL: Strategy B (Level 1 rho, then fidelity) ----------
def getrx_rz(x_1, x_0, z_1, z_0):
    rz = ((-1)*z_1 + z_0) / (z_0 + z_1)
    rx = ((-1)*x_1 + x_0) / (x_0 + x_1)

    norm = np.sqrt(rz**2 + rx**2)
    norm_factor = 1 / norm if norm > 1 else 1

    rz, rx = rz * norm_factor, rx * norm_factor
    return rx, rz


def fidelity_B(x_1, x_0, z_1, z_0):
    rx1, rz1 = getrx_rz(x_1, x_0, z_1, z_0)
    return 0.5*(1+((rx1 + rz1)/np.sqrt(2)))


# ---------- EXPERIMENT (grader) ----------
rT = np.array([1, 0, 1]) / np.sqrt(2)
rng = np.random.default_rng(seed=0)

for k in (0.9, 1.0):
    r_true = k * rT
    F_true = 0.5 * (1 + r_true @ rT)
    for N in (60, 600, 6000):
        n = N // 2
        F_hats = []
        for _ in range(1000):
            x_1, x_0 = measure(r_true, 0, n, rng)
            z_1, z_0 = measure(r_true, 2, n, rng)
            F_hats.append(fidelity_B(x_1, x_0, z_1, z_0))
        print(f"k={k} N={N:5d} | bias {np.mean(F_hats)-F_true:+.4f}  SD {np.std(F_hats):.4f}")

k=0.9 N=   60 | bias -0.0097  SD 0.0569
k=0.9 N=  600 | bias +0.0001  SD 0.0225
k=0.9 N= 6000 | bias -0.0001  SD 0.0070
k=1.0 N=   60 | bias -0.0271  SD 0.0391
k=1.0 N=  600 | bias -0.0087  SD 0.0119
k=1.0 N= 6000 | bias -0.0026  SD 0.0038


In [ ]:
import numpy as np

# Simulating the creating of our hidden state (before this, the qubit has a rotation matrix applied to its state vector that puts the idealized vector at the z-axis )
def measure_direct(r_true, n, rng):
    """Measure n fresh qubits along the rT axis. Returns the number of +1 outcomes."""
    rT = np.array([1, 0, 1]) / np.sqrt(2)
    p_plus = 0.5 * (1 + r_true @ rT)          # probability of +1 (equals F_true)
    return int(np.sum(rng.random(n) < p_plus))


# ---------- PROTOCOL: Strategy C (sees only the count) ----------
def fidelity_C(plus_count, n):
    return plus_count / n                      # fraction of +1s, already in [0, 1]


# ---------- EXPERIMENT (grader) ----------
rT = np.array([1, 0, 1]) / np.sqrt(2)
rng = np.random.default_rng(seed=0)

for k in (0.9, 1.0):
    r_true = k * rT
    F_true = 0.5 * (1 + r_true @ rT)
    for N in (60, 600, 6000):
        n = N                                  # all N shots on the one observable
        F_hats = []
        for _ in range(1000):
            plus = measure_direct(r_true, n, rng)
            F_hats.append(fidelity_C(plus, n))
        print(f"k={k} N={N:5d} | bias {np.mean(F_hats)-F_true:+.4f}  SD {np.std(F_hats):.4f}")

k=0.9 N=   60 | bias -0.0007  SD 0.0278
k=0.9 N=  600 | bias -0.0003  SD 0.0089
k=0.9 N= 6000 | bias +0.0001  SD 0.0028
k=1.0 N=   60 | bias +0.0000  SD 0.0000
k=1.0 N=  600 | bias +0.0000  SD 0.0000
k=1.0 N= 6000 | bias +0.0000  SD 0.0000


In [ ]:
import numpy as np

epsilon = 0.05                       # readout error: probability a reported outcome is flipped

# ---------- SIMULATOR (the only part that knows r_true and epsilon) ----------
def measure(r_true, axis, n, rng):
    """axis: 0 = X, 1 = Y, 2 = Z. Returns REPORTED (number of 1s, number of 0s)."""
    P1 = (1 - r_true[axis]) / 2
    true_one = rng.random(n) < P1                 # what the qubit actually measured as
    flip = rng.random(n) < epsilon                # separate random numbers produce the error
    ones = int(np.sum(true_one ^ flip))           # flipped outcomes swap 1 <-> 0
    return ones, n - ones

def calibrate(n, rng):
    #  Used as calibration, testing amount of errors when we measure the 0 state vector (should always collapse to 0 but error can flip it)
    return int(np.sum(rng.random(n) < epsilon))

def measure_direct(r_true, n, rng):
    """Measure n fresh qubits along the rT axis. Returns the number of REPORTED +1 outcomes."""
    rT = np.array([1, 0, 1]) / np.sqrt(2)
    p_plus = 0.5 * (1 + r_true @ rT)
    true_plus = rng.random(n) < p_plus
    flip = rng.random(n) < epsilon
    return int(np.sum(true_plus ^ flip))


# ---------- PROTOCOLS (see only counts, naive: they ignore readout error) ----------
c, s = np.cos(np.pi/8), np.sin(np.pi/8)
T = np.array([c, s])

def estimate_eps(flips, n_cal):
    return flips / n_cal if n_cal > 0 else 0.0    # 0 means "naive": no correction

def reconstruct_rho(x_1, x_0, y_1, y_0, z_1, z_0, eps):
    rz = ((-1)*z_1 + z_0) / ((z_0 + z_1) * (1-2*eps))
    ry = ((-1)*y_1 + y_0) / ((y_0 + y_1) * (1-2*eps))
    rx = ((-1)*x_1 + x_0) / ((x_0 + x_1) * (1-2*eps))
    norm = np.sqrt(rz**2 + rx**2 + ry**2)
    norm_factor = 1 / norm if norm > 1 else 1
    rz, ry, rx = rz * norm_factor, ry * norm_factor, rx * norm_factor
    return 0.5 * np.array([[1 + rz, rx - ry*(1j)], [rx + ry*(1j), 1 - rz]])

def fidelity_A(x_1, x_0, y_1, y_0, z_1, z_0, eps):
    p = reconstruct_rho(x_1, x_0, y_1, y_0, z_1, z_0, eps)
    return np.real(T.conj() @ p @ T)

def getrx_rz(x_1, x_0, z_1, z_0, eps):
    rz = ((-1)*z_1 + z_0) / ((z_0 + z_1) * (1-2*eps))
    rx = ((-1)*x_1 + x_0) / ((x_0 + x_1) * (1-2*eps))
    norm = np.sqrt(rz**2 + rx**2)
    norm_factor = 1 / norm if norm > 1 else 1
    return rx * norm_factor, rz * norm_factor

def fidelity_B(x_1, x_0, z_1, z_0, eps):
    rx, rz = getrx_rz(x_1, x_0, z_1, z_0, eps)
    return 0.5 * (1 + (rx + rz) / np.sqrt(2))

def fidelity_C(plus_count, n, eps):
    return np.clip(((plus_count / n) - eps)/(1-2*eps), 0, 1)


# ---------- EXPERIMENT (grader) ----------
rT = np.array([1, 0, 1]) / np.sqrt(2)
rng = np.random.default_rng(seed=0)
f_cal = 0.2

for k in (0.9, 1.0):
    r_true = k * rT
    F_true = 0.5 * (1 + r_true @ rT)
    for N in (600, 6000):                          # divisible by 2 and 3
        n_cal = int(f_cal * N)
        F_A, F_B, F_C = [], [], []
        Nstate = N - n_cal
        for _ in range(1000):
            flips = calibrate(n_cal, rng)
            eps_est = estimate_eps(flips, n_cal)
            n = Nstate // 3                             # A: N/3 on X, Y, Z
            x_1, x_0 = measure(r_true, 0, n, rng)
            y_1, y_0 = measure(r_true, 1, n, rng)
            z_1, z_0 = measure(r_true, 2, n, rng)
            F_A.append(fidelity_A(x_1, x_0, y_1, y_0, z_1, z_0, eps_est))

            n = Nstate // 2                             # B: N/2 on X and Z
            x_1, x_0 = measure(r_true, 0, n, rng)
            z_1, z_0 = measure(r_true, 2, n, rng)
            F_B.append(fidelity_B(x_1, x_0, z_1, z_0, eps_est))

            F_C.append(fidelity_C(measure_direct(r_true, Nstate, rng), Nstate, eps_est))   # C: all N on rT axis
        for name, vals in (("A", F_A), ("B", F_B), ("C", F_C)):
            print(f"k={k} n_cal = {n_cal:5d} N ={N:5d} Nstate={Nstate:5d} {name} | bias {np.mean(vals)-F_true:+.4f}  SD {np.std(vals):.4f}")

k=0.9 n_cal =   120 N =  600 Nstate=  480 A | bias -0.0035  SD 0.0354
k=0.9 n_cal =   120 N =  600 Nstate=  480 B | bias -0.0014  SD 0.0315
k=0.9 n_cal =   120 N =  600 Nstate=  480 C | bias -0.0007  SD 0.0233
k=0.9 n_cal =  1200 N = 6000 Nstate= 4800 A | bias +0.0003  SD 0.0127
k=0.9 n_cal =  1200 N = 6000 Nstate= 4800 B | bias +0.0007  SD 0.0116
k=0.9 n_cal =  1200 N = 6000 Nstate= 4800 C | bias +0.0005  SD 0.0081
k=1.0 n_cal =   120 N =  600 Nstate=  480 A | bias -0.0169  SD 0.0215
k=1.0 n_cal =   120 N =  600 Nstate=  480 B | bias -0.0135  SD 0.0202
k=1.0 n_cal =   120 N =  600 Nstate=  480 C | bias -0.0091  SD 0.0127
k=1.0 n_cal =  1200 N = 6000 Nstate= 4800 A | bias -0.0054  SD 0.0077
k=1.0 n_cal =  1200 N = 6000 Nstate= 4800 B | bias -0.0043  SD 0.0063
k=1.0 n_cal =  1200 N = 6000 Nstate= 4800 C | bias -0.0029  SD 0.0042
